# Recitation 8: Neural Networks (MLP) with scikit-learn
### Data Science for Entrepreneurship-Economy | Spring 2026

**Today's goal**: Build your first neural network and compare it against the traditional classifiers from Recitation 3.

By the end of this session you'll be able to:
- Train an MLP classifier using scikit-learn's familiar API
- Choose hidden layer sizes and activation functions
- Diagnose overfitting in neural networks
- Compare MLP vs. Decision Tree / Logistic Regression
- Decide when a neural network is worth the complexity

**Time**: ~90 minutes | **Exercises**: 3 | **Discussions**: 3 | **Format**: Guided walkthrough

**The big question**: Lecture 8 showed that MLPs learn feature interactions automatically. Today we test that claim with real data.

**A note on tools**: We use scikit-learn's `MLPClassifier`, which follows the same `create → .fit() → .predict()` API you already know. sklearn's MLP runs on CPU only and offers basic architectures — for production deep learning on large datasets, frameworks like TensorFlow/Keras or PyTorch provide GPU acceleration and much more flexibility. Those are beyond this course's scope, but the concepts you learn today transfer directly.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier       # the star of today's show
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

sns.set_style("whitegrid")

In [ ]:
df = pd.read_csv("spotpark_parking.csv")
print(f"Dataset: {df.shape[0]} observations × {df.shape[1]} columns")
print(f"Target balance: {df['zone_status'].value_counts().to_dict()}")
print(f"  → {df['zone_status'].mean():.1%} full\n")
df.head()

> **SpotPark** is an Israeli smart-parking startup operating in Tel Aviv. The app helps drivers find available street parking by predicting whether a given parking zone will have free spots at a requested time. Currently, drivers waste an average of 18 minutes circling for parking — costing time, fuel, and emissions. SpotPark's data scientist, Rotem, has collected 5,000 historical records of parking zone status (available / full) along with contextual features. She wants a model that predicts parking availability so the app can route drivers toward zones with open spots.
>
> **Rotem's challenge**: predict parking availability from contextual features. The catch — feature INTERACTIONS drive parking patterns more than individual features.

In [ ]:
# Show why interactions matter
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: restaurant count × hour
pivot = df.pivot_table(values="zone_status", index="hour",
                       columns=pd.cut(df["nearby_restaurants"], bins=[-1, 5, 15, 30],
                                       labels=["Few (0-5)", "Some (6-15)", "Many (16-30)"]),
                       aggfunc="mean")
pivot.plot(ax=axes[0], marker="o")
axes[0].set_ylabel("Proportion Full")
axes[0].set_title("Restaurants × Hour → Parking Status")
axes[0].legend(title="Restaurants nearby")

# Right: rain × weekend
rain_weekend = df.groupby(["rain", "is_weekend"])["zone_status"].mean().unstack()
rain_weekend.plot(kind="bar", ax=axes[1], color=["#214073", "#E6592D"])
axes[1].set_title("Rain × Weekend → Parking Status")
axes[1].set_xticklabels(["No Rain", "Rain"], rotation=0)
axes[1].legend(["Weekday", "Weekend"])
axes[1].set_ylabel("Proportion Full")

plt.tight_layout()
plt.show()

### 📖 The Interaction Story

Look at the left chart: zones with many restaurants are available in the morning but packed in the evening. A linear model sees "more restaurants → more full" — but the TRUTH is "more restaurants + evening → full; more restaurants + morning → fine." That's an **interaction**. The MLP should capture it.

In [ ]:
X = df.drop(columns=["zone_status", "observation_id"])
y = df["zone_status"]

# Scale features (important for MLP — same reason as KNN in Rec 3)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

In [ ]:
# Build the same baselines from Recitation 3
baselines = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree (d=5)": DecisionTreeClassifier(max_depth=5, random_state=42),
}

print(f"{'Model':<25} {'Accuracy':<12} {'F1 (Full)':<12}")
print("-" * 49)
baseline_results = {}
for name, model in baselines.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    f1_full = report["1"]["f1-score"]
    baseline_results[name] = {"accuracy": acc, "f1_full": f1_full, "y_pred": y_pred}
    print(f"{name:<25} {acc:<12.3f} {f1_full:<12.3f}")

### 💡 Insight

Decision Tree is the best traditional model at ~77% accuracy. Not bad — but Rotem knows the interactions in her data are complex. Can an MLP do better by learning those interactions automatically?

---
## 🧠 Section 2: Your First MLP
*Same API. Same `.fit()`. Same `.predict()`. But inside — neurons, layers, and learned features.*

---

### 📖 Recap

In Lecture 8 you learned that an MLP has: an **input layer** (your features), one or more **hidden layers** (where it learns interactions), and an **output layer** (the prediction). Each neuron computes a weighted sum + activation function. Training adjusts the weights to minimize prediction error.

sklearn's `MLPClassifier` wraps all of this in the familiar API.

In [ ]:
# MLP with 1 hidden layer of 32 neurons
mlp_simple = MLPClassifier(
    hidden_layer_sizes=(32,),        # one hidden layer, 32 neurons
    activation="relu",                # ReLU for hidden layers (from Lecture 8)
    max_iter=500,                     # maximum training epochs
    random_state=42,
)
mlp_simple.fit(X_train, y_train)
y_pred_mlp = mlp_simple.predict(X_test)

acc_mlp = accuracy_score(y_test, y_pred_mlp)
print(f"MLP (32 neurons, 1 layer):")
print(f"  Accuracy: {acc_mlp:.3f}")
print(f"  vs. Decision Tree: {baseline_results['Decision Tree (d=5)']['accuracy']:.3f}")

In [ ]:
print(classification_report(y_test, y_pred_mlp,
      target_names=["Available", "Full"]))

In [ ]:
print(f"Architecture: {X_train.shape[1]} inputs → 32 hidden → 2 outputs")
print(f"Number of weight matrices: {len(mlp_simple.coefs_)}")
print(f"  Layer 1 weights shape: {mlp_simple.coefs_[0].shape}")
print(f"  Layer 2 weights shape: {mlp_simple.coefs_[1].shape}")
total_params = sum(w.size for w in mlp_simple.coefs_) + sum(b.size for b in mlp_simple.intercepts_)
print(f"Total parameters: {total_params}")
print(f"\nTraining iterations: {mlp_simple.n_iter_}")
print(f"Final loss: {mlp_simple.loss_:.4f}")

A simple MLP with 32 hidden neurons has ~350 parameters (weights + biases). Compare: a Logistic Regression has ~9 parameters (one per feature). More parameters = more capacity to learn interactions — but also more risk of overfitting.

### ⚠️ Common Mistake

If you forget to scale features before MLP, performance drops dramatically. Neural networks are sensitive to feature scales — just like KNN in Recitation 3. `StandardScaler` is non-negotiable.

### 🎯 Exercise 1: Understand the Architecture

The MLP has `hidden_layer_sizes=(32,)`. That means 1 hidden layer with 32 neurons.

- How many weight connections are there between the input (8 features) and the hidden layer? *(Hint: every input connects to every hidden neuron)*
- How many between the hidden layer (32 neurons) and the output (2 classes)?
- Does the total match `mlp_simple.coefs_` shapes from above?
- Why does more parameters = more capacity to learn? *(Think back to Lecture 8's analogy)*

In [ ]:
##### Your code goes here #####

# Input → Hidden connections: 8 × 32 = ?
# Hidden → Output connections: 32 × 2 = ?
# Total weight parameters: ? (don't forget biases: 32 + 2)

# Why more parameters = more capacity? (comment)

###### End of Your Code ######

<details><summary>✅ Solution</summary>

```python
# Input → Hidden: 8 × 32 = 256
# Hidden → Output: 32 × 2 = 64
# Total weights: 256 + 64 = 320
# Plus biases: 32 + 2 = 34
# Grand total: 320 + 34 = 354 parameters

print(f"Input → Hidden: {8 * 32}")
print(f"Hidden → Output: {32 * 2}")
print(f"Biases: {32 + 2}")
print(f"Total: {8 * 32 + 32 * 2 + 32 + 2}")
```

More parameters means the network can represent more complex decision boundaries — it has more "knobs to tune." But more knobs also means it can memorize noise (overfitting) if data is insufficient.
</details>

---
## 🏗️ Section 3: Architecture Exploration
*How many layers? How many neurons? The architecture IS the design decision.*

---

### 📖 Recap

In Lecture 8 you learned that hidden layers create new features: layer 1 might learn "rain + evening," layer 2 might combine that with "many restaurants." More layers = more abstract combinations. But more layers also means more parameters and more overfitting risk. Let's test different architectures.

In [ ]:
architectures = {
    "(16,)":         (16,),
    "(32,)":         (32,),
    "(64,)":         (64,),
    "(32, 16)":      (32, 16),
    "(64, 32)":      (64, 32),
    "(64, 32, 16)":  (64, 32, 16),
}

print(f"{'Architecture':<20} {'Accuracy':<12} {'F1 (Full)':<12} {'Parameters':<12}")
print("-" * 56)

arch_results = {}
for name, layers in architectures.items():
    mlp = MLPClassifier(hidden_layer_sizes=layers, activation="relu",
                        max_iter=500, random_state=42)
    mlp.fit(X_train, y_train)
    y_p = mlp.predict(X_test)
    acc = accuracy_score(y_test, y_p)
    report = classification_report(y_test, y_p, output_dict=True)
    f1 = report["1"]["f1-score"]
    n_params = sum(w.size for w in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)
    arch_results[name] = {"accuracy": acc, "f1_full": f1, "n_params": n_params, "y_pred": y_p}
    print(f"{name:<20} {acc:<12.3f} {f1:<12.3f} {n_params:<12d}")

In [ ]:
names = list(arch_results.keys())
accs = [arch_results[n]["accuracy"] for n in names]
params = [arch_results[n]["n_params"] for n in names]

fig, ax1 = plt.subplots(figsize=(10, 5))
x = range(len(names))
ax1.bar(x, accs, color="#214073", alpha=0.7, label="Accuracy")
ax1.set_ylabel("Accuracy")
ax1.set_ylim(0.65, 0.90)
ax1.set_xticks(x)
ax1.set_xticklabels(names, rotation=30)

ax2 = ax1.twinx()
ax2.plot(x, params, "o-", color="#E6592D", linewidth=2, label="Parameters")
ax2.set_ylabel("Number of Parameters")

ax1.legend(loc="upper left")
ax2.legend(loc="upper right")
plt.title("Architecture Size vs. Accuracy — Diminishing Returns")
plt.tight_layout()
plt.show()

Notice the diminishing returns: going from (16,) to (32,16) improves accuracy noticeably. But going from (32,16) to (64,32,16) barely helps — while tripling the parameter count. More complexity doesn't always mean better predictions.

### 💬 Discussion 1: Pick the Architecture *(Instructor-Led)*

Look at the table and chart above. Let's discuss as a class:
- Which architecture gives the best accuracy?
- Which gives the best accuracy-per-parameter (best performance with fewest parameters)?
- Rotem's app runs on users' phones with limited compute. Which architecture would you recommend and why?
- Is the 3-layer architecture worth the extra complexity? Why or why not?

<details><summary>✅ Discussion Points</summary>

(32, 16) likely gives the best accuracy-per-parameter tradeoff. For mobile: (32, 16) — small enough for phone inference, accurate enough to be useful. The 3-layer network adds ~3x parameters for marginal gain — classic diminishing returns.
</details>

### 💡 Insight

Architecture design is the ONE thing the human still controls in neural networks. The model learns the weights — but YOU choose how many layers and neurons. With sklearn's `MLPClassifier`, this is just a tuple: `(32, 16)` means 2 layers with 32 and 16 neurons.

---
## ⚠️ Section 4: Overfitting — When the Network Memorizes
*More capacity without more data = trouble. Neural networks overfit harder than anything you've seen.*

---

In [ ]:
# Deliberately overfit: huge network, many iterations
mlp_overfit = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),   # massive for our dataset
    activation="relu",
    max_iter=2000,
    random_state=42,
)
mlp_overfit.fit(X_train, y_train)

train_acc = mlp_overfit.score(X_train, y_train)
test_acc = mlp_overfit.score(X_test, y_test)

print(f"Overfit MLP (256, 128, 64):")
print(f"  Train accuracy: {train_acc:.3f}")
print(f"  Test accuracy:  {test_acc:.3f}")
print(f"  Gap:            {train_acc - test_acc:.3f}")
n_params = sum(w.size for w in mlp_overfit.coefs_) + sum(b.size for b in mlp_overfit.intercepts_)
print(f"  Parameters:     {n_params:,}")

In [ ]:
# sklearn stores the loss at each iteration
plt.figure(figsize=(10, 5))
plt.plot(mlp_overfit.loss_curve_, color="#E6592D", linewidth=2)
plt.xlabel("Iteration (Epoch)")
plt.ylabel("Training Loss")
plt.title("Training Loss Curve — Overfit MLP")
plt.grid(True, alpha=0.3)
plt.show()
print("Training loss keeps dropping — but test accuracy doesn't improve.")
print("The model is memorizing the training data, not learning general patterns.")

### ⚠️ Common Mistake

A high training accuracy with a much lower test accuracy is classic **overfitting**. The large (256, 128, 64) network has tens of thousands of parameters for only 4,000 training samples. The model has enough capacity to memorize every training example, including the noise.

In [ ]:
# early_stopping=True monitors validation loss and stops when it starts rising
mlp_earlystop = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    max_iter=2000,
    early_stopping=True,          # the fix
    validation_fraction=0.15,     # use 15% of training data for validation
    n_iter_no_change=20,          # stop if no improvement for 20 iterations
    random_state=42,
)
mlp_earlystop.fit(X_train, y_train)

print(f"MLP with early stopping:")
print(f"  Stopped at iteration: {mlp_earlystop.n_iter_}")
print(f"  Train accuracy: {mlp_earlystop.score(X_train, y_train):.3f}")
print(f"  Test accuracy:  {mlp_earlystop.score(X_test, y_test):.3f}")

### 🎯 Exercise 2: Diagnose Overfitting

Compare three models: the overfit MLP (256,128,64), the early-stopped MLP (64,32), and the simple MLP (32,) from Section 2.
- For each, print train accuracy AND test accuracy
- Which has the smallest train-test gap?
- Which has the best TEST accuracy? (Test is what matters — not train)
- What's the lesson about model size vs. dataset size?

In [ ]:
##### Your code goes here #####

# Print train + test accuracy for all three

# Smallest gap:
# Best test accuracy:
# Lesson about size vs data: (comment)

###### End of Your Code ######

<details><summary>✅ Solution</summary>

```python
models_check = {
    "MLP (32,) simple": mlp_simple,
    "MLP (64,32) early stop": mlp_earlystop,
    "MLP (256,128,64) overfit": mlp_overfit,
}

print(f"{'Model':<30} {'Train Acc':<12} {'Test Acc':<12} {'Gap':<10}")
print("-" * 64)
for name, m in models_check.items():
    tr = m.score(X_train, y_train)
    te = m.score(X_test, y_test)
    print(f"{name:<30} {tr:<12.3f} {te:<12.3f} {tr - te:<10.3f}")
```

The simple or early-stopped MLP likely has the best test accuracy. The overfit MLP has the highest train accuracy but worst test accuracy. Lesson: more parameters need more data. Early stopping is your seatbelt.
</details>

### 💡 Insight

Neural networks are powerful but greedy — they'll eat all the data AND the noise if you let them. Early stopping is your seatbelt. For Rotem's dataset, a modest architecture with early stopping is the responsible choice.

---
## 🏆 Section 5: The Grand Comparison
*MLP vs. the Recitation 3 classics. Same data, same evaluation, different models.*

---

In [ ]:
# Use the best architecture from Section 3
mlp_best = MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu",
                          max_iter=500, random_state=42)
mlp_best.fit(X_train, y_train)
y_pred_best_mlp = mlp_best.predict(X_test)
report_mlp = classification_report(y_test, y_pred_best_mlp, output_dict=True)

all_results = []
for name, res in baseline_results.items():
    all_results.append({"Model": name, "Accuracy": res["accuracy"], "F1 (Full)": res["f1_full"]})

all_results.append({
    "Model": "MLP (32, 16)",
    "Accuracy": accuracy_score(y_test, y_pred_best_mlp),
    "F1 (Full)": report_mlp["1"]["f1-score"],
})

results_df = pd.DataFrame(all_results).set_index("Model").round(3)
results_df

In [ ]:
results_df.plot(kind="barh", figsize=(10, 4), color=["#214073", "#E6592D"])
plt.title("Model Comparison — Parking Zone Classification")
plt.xlabel("Score")
plt.xlim(0.5, 0.90)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
models_compare = [
    ("Logistic Reg.", baseline_results["Logistic Regression"]["y_pred"]),
    ("Decision Tree", baseline_results["Decision Tree (d=5)"]["y_pred"]),
    ("MLP (32, 16)", y_pred_best_mlp),
]

for ax, (name, preds) in zip(axes, models_compare):
    cm = confusion_matrix(y_test, preds)
    ConfusionMatrixDisplay(cm, display_labels=["Available", "Full"]).plot(ax=ax, cmap="Blues")
    ax.set_title(name, fontsize=10)

plt.suptitle("Confusion Matrices — All Models", fontsize=13)
plt.tight_layout()
plt.show()

### 💬 Discussion 2: Where Does the MLP Win? *(Instructor-Led)*

Look at the comparison table and confusion matrices. Let's discuss as a class:
- On which class (Available or Full) does the MLP improve most over Decision Tree?
- Why might the MLP be better at predicting "Full"? *(Think about the feature interactions from Section 1's EDA)*
- Is the accuracy improvement worth the added complexity of a neural network? Consider Rotem's context.

<details><summary>✅ Discussion Points</summary>

MLP likely improves most on "Full" — because "Full" is driven by complex interactions (restaurants × evening, rain × rush hour) that linear models miss. Each correct prediction saves a driver ~18 minutes of circling. At scale (thousands of daily predictions), the improvement is significant. Worth the complexity for a production app.
</details>

### 🎯 Exercise 3: Cross-Validation Check

Run 5-fold cross-validation on the best MLP architecture and on Decision Tree.
- Compare the mean ± std of accuracy
- Is the MLP advantage consistent across folds, or just lucky on one split?

In [ ]:
##### Your code goes here #####

# 5-fold CV for Decision Tree

# 5-fold CV for MLP (best architecture)

# Is the advantage consistent? (comment)

###### End of Your Code ######

<details><summary>✅ Solution</summary>

```python
dt_cv = cross_val_score(DecisionTreeClassifier(max_depth=5, random_state=42),
                        X_scaled, y, cv=5, scoring="accuracy")
mlp_cv = cross_val_score(MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu",
                                        max_iter=500, random_state=42),
                         X_scaled, y, cv=5, scoring="accuracy")

print(f"Decision Tree:  {dt_cv.mean():.3f} ± {dt_cv.std():.3f}")
print(f"MLP (32, 16):   {mlp_cv.mean():.3f} ± {mlp_cv.std():.3f}")
```

MLP should show consistently higher accuracy across folds (with slightly higher variance). If the advantage holds across 5 folds, it's real — not luck. Note: `cross_val_score` with MLP may be slow on CPU.
</details>

---
## 🤔 Section 6: When to Go Deep
*The MLP won today. But it doesn't always win. Know when to use it.*

---

### 📖 The Decision Framework

From Lecture 8, the deep learning decision checklist:

1. **Did you try simpler models first?** Always start with a baseline (Logistic Regression, Decision Tree). If they're good enough, stop.
2. **Do you have enough data?** With sklearn's CPU-only MLP, even a few thousand rows can work. Production deep learning frameworks (TensorFlow, PyTorch) with GPU can handle millions — but that's beyond our tools.
3. **Are there complex feature interactions?** If a linear model works well, the MLP won't add much. If interactions drive the outcome (like parking), the MLP shines.
4. **Can you tolerate a black box?** The MLP doesn't give you feature importance or interpretable rules. If Rotem needs to explain WHY a zone is predicted "Full," a Decision Tree is better.
5. **Is inference speed acceptable?** For a phone app predicting in real-time, a small MLP is fast. A huge network would be too slow without GPU.

### 💬 Discussion 3: Rotem's Final Recommendation *(Instructor-Led)*

> Rotem presents to SpotPark's CTO. Let's discuss her recommendation as a class:
>
> 1. **Which model to deploy** and why? (State the accuracy and F1)
> 2. **What the MLP learned** that traditional models couldn't (reference the feature interactions)
> 3. **The tradeoff** she accepted by choosing MLP over Decision Tree (black box, compute)
> 4. **A fallback plan**: If the MLP breaks in production, what should the system fall back to?
> 5. **Limitations of our approach**: What would Rotem need to use production deep learning frameworks (TensorFlow/PyTorch) instead of sklearn? (Hint: GPU, larger data, more flexible architectures)

<details><summary>✅ Discussion Points</summary>

1. **Deploy** the best MLP architecture with early stopping. Best performance of all tested models on both accuracy and F1.
2. **Interactions**: The MLP captured restaurant × hour interactions and rain × weekend × hour patterns that the Decision Tree partially missed and Logistic Regression completely missed.
3. **Tradeoff**: MLP is a black box — Rotem can't explain "why this zone is predicted Full" with simple rules. A Decision Tree with its splits was more interpretable. For the app's use case (just show "Available" or "Full"), black-box is acceptable.
4. **Fallback**: If MLP fails (e.g., convergence issues, latency), fall back to Decision Tree. It's the best traditional model and has no convergence risk.
5. **Sklearn limitations**: sklearn's MLP runs on CPU only — with 5,000 rows this is fine (seconds to train), but with millions of rows or image/text data, training would take hours. TensorFlow/Keras or PyTorch provide GPU acceleration, support for CNNs (images) and RNNs (text), and much more flexible architectures. For SpotPark at scale, migrating to a production framework would be the next step.
</details>

---

### 🎓 What You Just Did

- Built your first neural network using scikit-learn's `MLPClassifier`
- Compared MLP against Decision Tree and Logistic Regression on the same data
- Explored architecture choices: layers, neurons, activation functions
- Demonstrated and diagnosed overfitting — then fixed it with early stopping
- Showed that MLP captures feature interactions that traditional models miss
- Made an informed deploy-vs-don't-deploy recommendation

### The MLP in sklearn — Three Lines

```python
mlp = MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu",
                    early_stopping=True, random_state=42)
mlp.fit(X_train, y_train)
y_pred = mlp.predict(X_test)
```

Same `create → .fit() → .predict()` you've used since Recitation 3. The API is familiar — the power inside is new.

### A Note on What Comes Next

sklearn's `MLPClassifier` is a great learning tool but a limited production tool. It runs on CPU, supports only fully-connected layers, and lacks the flexibility of dedicated deep learning frameworks. For larger datasets, image recognition, text understanding, or any serious deep learning project, you'd use:

- **TensorFlow / Keras** — the industry standard, Google-backed
- **PyTorch** — research favorite, Meta-backed, very Pythonic
- Both support GPU acceleration, convolutional layers, recurrent layers, transformers, and much more

The concepts you learned today — layers, neurons, activation functions, loss, gradient descent, overfitting, early stopping — transfer directly to these frameworks. The syntax changes; the thinking doesn't.

### 📅 What's Next

- **Final Project**: Apply everything from Lectures 1–8 to a real business problem

### 💬 One Last Thing

"Eight recitations ago you typed `2 + 2` in a Colab cell. Today you built a neural network that predicts parking availability by learning feature interactions no human explicitly programmed. In between, you explored data, classified customers, predicted prices, segmented subscribers, built recommendation engines, processed text and images, and now — trained a brain-inspired model. That's not a semester of exercises. That's a data science education."